# FAISS Flat y persistencia

## Problema real

El índice tiene que sobrevivir a que reinicies el programa

## Conceptos clave

`IndexFlatIP` de FAISS hace lo mismo que el k-NN exacto de la notebook anterior, pero con una librería optimizada en C++. Si normalizás los embeddings antes de indexarlos, el producto interno (IP) se comporta igual que la similitud coseno. Lo importante: el índice y la metadata (qué `chunk_id` corresponde a cada posición) se guardan **juntos** -si no, perdés la trazabilidad.

In [ ]:
# FAISS devuelve posiciones dentro del indice, no el texto: por eso guardamos aparte
# que posicion corresponde a que chunk_id de SoporteBot.
posicion_a_chunk = {0: "facturacion-003", 1: "red-008", 2: "seguridad-011"}
scores = [.91, .77, .52]

for posicion, score in enumerate(scores):
    print({"posicion_faiss": posicion, "chunk_id": posicion_a_chunk[posicion], "score": score})

## Por qué guardar la metadata aparte

FAISS solo sabe de vectores y posiciones (0, 1, 2, ...). Si no guardás qué `chunk_id` corresponde a cada posición, cuando el índice te devuelva "posición 1" no vas a poder decirle al usuario de qué FAQ salió esa respuesta.

## Errores comunes

Guardar el índice de FAISS pero no la metadata (o al revés): si reiniciás el servidor y cargás solo uno de los dos, las posiciones ya no significan nada.

## El flujo completo, de punta a punta

Así se ve el camino completo para dejar un índice de FAISS listo para usar en producción.

In [ ]:
import matplotlib.pyplot as plt

pasos = ["cargar FAQs", "generar embeddings", "normalizar", "IndexFlatIP", "guardar indice + metadata"]
plt.figure(figsize=(9, 2.3))
plt.plot(range(len(pasos)), [0] * len(pasos), "o-", color="#1f4e79")
for i, paso in enumerate(pasos):
    plt.text(i, .1, paso, ha="center")
plt.axis("off")
plt.title("De las FAQs a un indice FAISS reproducible")
plt.show()